# S6.1 — Agent State & Runtime Context

Interactive verification of the foundational data structures for the LangGraph agent system:
- **AgentState** — TypedDict flowing through all nodes
- **AgentContext** — Per-request runtime context (dataclass)
- **Structured output models** — Pydantic models for LLM responses

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.agents import (
    AgentState, AgentContext, GuardrailScoring, GradeDocuments,
    GradingResult, SourceItem, RoutingDecision,
    create_initial_state, create_agent_context,
)
print("\u2713 All imports successful")

✓ All imports successful


## 1. Structured Output Models (FR-3)

In [2]:
from pydantic import ValidationError

# GuardrailScoring — valid
gs = GuardrailScoring(score=75, reason="Relevant to ML research")
assert gs.score == 75 and gs.reason == "Relevant to ML research"
print(f"GuardrailScoring: score={gs.score}, reason='{gs.reason}'")

# GuardrailScoring — boundary values
gs_zero = GuardrailScoring(score=0, reason="off-topic")
gs_max = GuardrailScoring(score=100, reason="perfect match")
assert gs_zero.score == 0 and gs_max.score == 100

# GuardrailScoring — invalid score rejected
try:
    GuardrailScoring(score=101, reason="invalid")
    assert False, "Should have raised"
except ValidationError:
    print("GuardrailScoring: score=101 correctly rejected")

# GradeDocuments
gd_yes = GradeDocuments(binary_score="yes", reasoning="contains relevant info")
gd_no = GradeDocuments(binary_score="no")
assert gd_yes.binary_score == "yes" and gd_no.reasoning == ""
try:
    GradeDocuments(binary_score="maybe")
    assert False
except ValidationError:
    print("GradeDocuments: 'maybe' correctly rejected")

# SourceItem
si = SourceItem(
    arxiv_id="1706.03762",
    title="Attention Is All You Need",
    authors=["Vaswani", "Shazeer"],
    url="https://arxiv.org/abs/1706.03762",
    relevance_score=0.95,
    chunk_text="The dominant sequence transduction models...",
)
assert si.authors == ["Vaswani", "Shazeer"]
print(f"SourceItem: {si.title} (score={si.relevance_score})")
print(f"  Serialized keys: {list(si.model_dump().keys())}")

# RoutingDecision — all valid routes
for route in ["retrieve", "out_of_scope", "generate_answer", "rewrite_query"]:
    rd = RoutingDecision(route=route, reason="test")
    assert rd.route == route
try:
    RoutingDecision(route="invalid")
    assert False
except ValidationError:
    print("RoutingDecision: 'invalid' route correctly rejected")

# GradingResult
gr = GradingResult(document_id="doc-1", is_relevant=True, score=0.9, reasoning="On topic")
assert gr.is_relevant and gr.score == 0.9
print(f"GradingResult: doc={gr.document_id}, relevant={gr.is_relevant}, score={gr.score}")

print("\n\u2713 All structured output models validated")

GuardrailScoring: score=75, reason='Relevant to ML research'
GuardrailScoring: score=101 correctly rejected
GradeDocuments: 'maybe' correctly rejected
SourceItem: Attention Is All You Need (score=0.95)
  Serialized keys: ['arxiv_id', 'title', 'authors', 'url', 'relevance_score', 'chunk_text']
RoutingDecision: 'invalid' route correctly rejected
GradingResult: doc=doc-1, relevant=True, score=0.9

✓ All structured output models validated


## 2. AgentState TypedDict (FR-1, FR-4)

In [3]:
import typing
from typing import Annotated, get_type_hints
from langchain_core.messages import HumanMessage
from langgraph.graph import add_messages

# Verify TypedDict structure
assert hasattr(AgentState, "__required_keys__")
assert len(AgentState.__required_keys__) == 0, "total=False means no required keys"
print(f"AgentState fields: {list(AgentState.__annotations__.keys())}")
print(f"Required keys: {AgentState.__required_keys__} (empty = total=False)")

# Verify add_messages reducer on messages field
hints = get_type_hints(AgentState, include_extras=True)
messages_hint = hints["messages"]
assert typing.get_origin(messages_hint) is Annotated
assert add_messages in typing.get_args(messages_hint)
print("messages field uses Annotated[..., add_messages] reducer")

# Create initial state
state = create_initial_state("What are the latest advances in transformer architectures?")
assert len(state["messages"]) == 1
assert isinstance(state["messages"][0], HumanMessage)
assert state["original_query"] == "What are the latest advances in transformer architectures?"
assert state["retrieval_attempts"] == 0
assert state["sources"] == []
print(f"\nInitial state created:")
print(f"  messages: [{type(state['messages'][0]).__name__}('{state['messages'][0].content[:50]}...')]")
print(f"  original_query: '{state['original_query'][:50]}...'")
print(f"  retrieval_attempts: {state['retrieval_attempts']}")
print(f"  routing_decision: {state['routing_decision']}")

# Empty query raises ValueError
try:
    create_initial_state("")
    assert False
except ValueError as e:
    print(f"\nEmpty query correctly raises: {e}")

# Partial returns work (simulates node output)
partial = {"routing_decision": "retrieve", "retrieval_attempts": 1}
assert partial["routing_decision"] == "retrieve"
print(f"\nPartial return works: {partial}")

print("\n\u2713 AgentState and create_initial_state validated")

AgentState fields: ['messages', 'original_query', 'rewritten_query', 'retrieval_attempts', 'guardrail_result', 'routing_decision', 'sources', 'grading_results', 'relevant_sources', 'metadata']
Required keys: frozenset() (empty = total=False)
messages field uses Annotated[..., add_messages] reducer

Initial state created:
  messages: [HumanMessage('What are the latest advances in transformer archit...')]
  original_query: 'What are the latest advances in transformer archit...'
  retrieval_attempts: 0
  routing_decision: None

Empty query correctly raises: query must be a non-empty string

Partial return works: {'routing_decision': 'retrieve', 'retrieval_attempts': 1}

✓ AgentState and create_initial_state validated


## 3. AgentContext Dataclass (FR-2, FR-5)

In [4]:
from dataclasses import fields
from collections.abc import AsyncIterator
from src.services.llm.provider import LLMProvider, LLMResponse, HealthStatus

# Mock provider for demo
class DemoProvider:
    async def generate(self, prompt, *, model=None, temperature=None, max_tokens=None):
        return LLMResponse(text="demo", model="demo", provider="demo")
    async def generate_stream(self, prompt, *, model=None, temperature=None, max_tokens=None):
        yield "demo"
    async def health_check(self):
        return HealthStatus(healthy=True, provider="demo", message="ok")
    def get_langchain_model(self, *, model=None, temperature=None):
        return None
    async def close(self):
        pass

provider = DemoProvider()
assert isinstance(provider, LLMProvider), "DemoProvider satisfies LLMProvider protocol"

# Create context with defaults
ctx = create_agent_context(llm_provider=provider)
assert ctx.llm_provider is provider
assert ctx.temperature == 0.7
assert ctx.top_k == 5
assert ctx.max_retrieval_attempts == 3
assert ctx.guardrail_threshold == 40
assert ctx.user_id == "api_user"
assert ctx.retrieval_pipeline is None
assert ctx.cache_service is None

print("AgentContext defaults:")
for f in fields(AgentContext):
    print(f"  {f.name}: {getattr(ctx, f.name)!r}")

# Create context with overrides
ctx2 = create_agent_context(
    llm_provider=provider,
    model_name="gemini-3-flash",
    temperature=0.3,
    top_k=10,
    user_id="researcher_1",
    trace_id="trace-abc-123",
)
assert ctx2.model_name == "gemini-3-flash"
assert ctx2.temperature == 0.3
assert ctx2.top_k == 10
assert ctx2.user_id == "researcher_1"
print(f"\nOverridden context: model={ctx2.model_name}, temp={ctx2.temperature}, top_k={ctx2.top_k}")

# Verify slots
assert hasattr(AgentContext, "__slots__")
print(f"\nAgentContext uses __slots__: {bool(AgentContext.__slots__)}")

print("\n\u2713 AgentContext and create_agent_context validated")

AgentContext defaults:
  llm_provider: <__main__.DemoProvider object at 0x10d35e1e0>
  retrieval_pipeline: None
  cache_service: None
  model_name: ''
  temperature: 0.7
  top_k: 5
  max_retrieval_attempts: 3
  guardrail_threshold: 40
  trace_id: None
  user_id: 'api_user'

Overridden context: model=gemini-3-flash, temp=0.3, top_k=10

AgentContext uses __slots__: True

✓ AgentContext and create_agent_context validated
